In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Allow the notebook to import modules from the project root.
sys.path.append("..")

from config import CONFIG, REQUIRED_COLUMNS
from src.data_processing import (
    load_csv_chunks,
    prepare_chunk,
)
from src.analysis import CustomsAnalyzer
from src.numpy_analysis import run_numpy_comparison
from src.visualization import (
    create_bar_plot,
    create_heatmap,
)
from src.validation import (
    inspect_raw_data,
    inspect_dataset,
    create_validation_results,
    calculate_file_sha256,
    save_validation_results,
)

In [ ]:
chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

first_chunk = next(chunks)

print("First chunk rows:", len(first_chunk))
print("Columns:")
print(first_chunk.columns.tolist())

In [ ]:
inspection = inspect_dataset(
    CONFIG["input_path"],
    CONFIG["chunksize"],
)

inspection

In [ ]:
raw_stats = inspect_raw_data(
    CONFIG["input_path"],
    CONFIG["chunksize"],
)

raw_stats

In [ ]:
analyzer = CustomsAnalyzer(
    output_dir=CONFIG["output_dir"],
    group_columns=CONFIG["group_columns"],
    measure_column=CONFIG["measure_column"],
)

chunks = load_csv_chunks(
    CONFIG["input_path"],
    REQUIRED_COLUMNS,
    CONFIG["chunksize"],
)

for chunk in chunks:
    selected = prepare_chunk(
        chunk,
        CONFIG["minimum_dutiable_value_php"],
        CONFIG["high_value_threshold_million_php"],
    )

    analyzer.add_chunk(
        selected,
        len(chunk),
        len(selected),
    )

data = analyzer.combine_chunks()

print("Selected rows:", len(data))
print()
print(data.head())

In [ ]:
data.loc[
    (
        data["tq"].notna()
        & (data["dutiablevaluephp"] > 0)
    ),
    [
        "tq",
        "countryorigin_iso3",
        "dutiablevaluephp",
        "dutiablevalue_million_php",
        "value_band",
    ],
].head(10)

In [ ]:
grouped = analyzer.create_grouped_summary(data)

grouped.head(10)

In [ ]:
print("Number of country groups:", len(grouped))
print("Total grouped rows:", grouped["row_count"].sum())
print("Total grouped measure:", grouped["measure_sum"].sum())

In [ ]:
grouped.to_csv(
    CONFIG["output_dir"] / "grouped.csv",
    index=False,
)

print("grouped.csv saved.")

In [ ]:
grouped_two = analyzer.create_two_category_summary(data)

grouped_two.head(10)

In [ ]:
print("Number of country-quarter groups:", len(grouped_two))
print("Total grouped rows:", grouped_two["row_count"].sum())
print("Total grouped measure:", grouped_two["measure_sum"].sum())

In [ ]:
grouped_two.to_csv(
    CONFIG["output_dir"] / "grouped_two.csv",
    index=False,
)

print("grouped_two.csv saved.")

In [ ]:
pivot = analyzer.create_pivot(grouped_two)

pivot.head(10)

In [ ]:
print("Pivot grand total:", pivot.loc[pivot["countryorigin_iso3"] == "Total", "Total"].iloc[0])

In [ ]:
pivot.to_csv(
    CONFIG["output_dir"] / "pivot.csv",
    index=False,
)

print("pivot.csv saved.")

In [ ]:
top10 = analyzer.create_top10(grouped)

top10

In [ ]:
print("Number of rows in top10:", len(top10))
print("Top 10 total measure:", top10["measure_sum"].sum())

In [ ]:
top10.to_csv(
    CONFIG["output_dir"] / "top10.csv",
    index=False,
)

print("top10.csv saved.")

In [ ]:
numpy_results = run_numpy_comparison(data)

numpy_results

In [ ]:
print("Loop and vectorized results agree:", numpy_results.attrs["numpy_equal"])

In [ ]:
numpy_results.to_csv(
    CONFIG["output_dir"] / "numpy_comparison.csv",
    index=False,
)

print("numpy_comparison.csv saved.")